Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
!pip3 install accelerate
!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip3 install torchsde
!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


Mounting Google Drive...
/
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
-= Updating ComfyUI =-
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
-= Install dependencies =-
Looking in indexes: https://download.pytorch.org/whl/cu121
/content/drive/MyDrive/ComfyUI/custom_nodes
/content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Manager
^C
/content/drive/MyDrive/ComfyUI
-= Install custom nodes dependencies =-

WARN: The `COMFYUI_PATH` environment variable is not set. Assuming 
`custom_nodes/ComfyUI-Manager/../../` as the ComfyUI path.
Traceback (most recent call last):
  File "/content/drive/MyDrive/ComfyUI/custom_nodes/ComfyUI-Manager/cm-cli.py", line 38, in <module>
    import utils.extra_config
ModuleNotFoundError: No module named 'utils'


Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [2]:
import os
print("DOWNLOADING SDXL MODELS TO GOOGLE DRIVE...")
COMFY_DIR = "/content/drive/MyDrive/ComfyUI/models"

os.makedirs(f"{COMFY_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{COMFY_DIR}/ipadapter", exist_ok=True)
os.makedirs(f"{COMFY_DIR}/controlnet", exist_ok=True)
os.makedirs(f"{COMFY_DIR}/loras", exist_ok=True)

# 1. SDXL Base checkpoint
os.system(f"wget -q --show-progress -c 'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors' -O {COMFY_DIR}/checkpoints/sd_xl_base_1.0.safetensors")
print("SDXL BASE DONE")

# 2. IP-Adapter SDXL (ViT-H)
os.system(f"wget -q --show-progress -c 'https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter_sdxl_vit-h.safetensors' -O {COMFY_DIR}/ipadapter/ip-adapter_sdxl_vit-h.safetensors")
print("IPADAPTER SDXL DONE")

# 3. SDXL ControlNet OpenPose
os.system(f"wget -q --show-progress -c 'https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/OpenPoseXL2.safetensors' -O {COMFY_DIR}/controlnet/OpenPoseXL2.safetensors")
print("CONTROLNET OP DONE")

# 4. Chhota Bheem LoRA (SDXL trained)
os.system(f"wget -q --show-progress -c 'https://huggingface.co/Kousha/chhota_bheem_LoRA/resolve/main/pytorch_lora_weights.safetensors' -O {COMFY_DIR}/loras/chhota_bheem_sdxl.safetensors")
print("BHEEM LORA DONE")

# 5. IP-Adapter FaceID SDXL
os.system(f"wget -q --show-progress -c 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid_sdxl.bin' -O {COMFY_DIR}/ipadapter/ip-adapter-faceid_sdxl.bin")
print("FACEID SDXL DONE")

# 6. Depth SDXL
os.system(f"wget -q --show-progress -c 'https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors' -O {COMFY_DIR}/controlnet/depth_sdxl.safetensors")
print("DEPTH SDXL DONE")

# 7. Canny SDXL
os.system(f"wget -q --show-progress -c 'https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors' -O {COMFY_DIR}/controlnet/canny_sdxl.safetensors")
print("CANNY SDXL DONE")

print("ALL_EXTRA_DONE")


DOWNLOADING SDXL MODELS TO GOOGLE DRIVE...
SDXL BASE DONE
IPADAPTER SDXL DONE
CONTROLNET OP DONE
BHEEM LORA DONE
FACEID SDXL DONE
DEPTH SDXL DONE
CANNY SDXL DONE
ALL_EXTRA_DONE


In [3]:
# Checkpoints

### SDXL
### I recommend these workflow examples: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/

#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors -P ./models/checkpoints/

# SDXL ReVision
#!wget -c https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors -P ./models/clip_vision/

# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# SD2
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors -P ./models/checkpoints/

# Some SD1.5 anime style
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A3_orangemixs.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/Linaqruf/anything-v3.0/resolve/main/anything-v3-fp16-pruned.safetensors -P ./models/checkpoints/

# Waifu Diffusion 1.5 (anime style SD2.x 768-v)
#!wget -c https://huggingface.co/waifu-diffusion/wd-1-5-beta3/resolve/main/wd-illusion-fp16.safetensors -P ./models/checkpoints/


# unCLIP models
#!wget -c https://huggingface.co/comfyanonymous/illuminatiDiffusionV1_v11_unCLIP/resolve/main/illuminatiDiffusionV1_v11-unclip-h-fp16.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/comfyanonymous/wd-1.5-beta2_unCLIP/resolve/main/wd-1-5-beta2-aesthetic-unclip-h-fp16.safetensors -P ./models/checkpoints/


# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
#!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt -P ./models/vae/
#!wget -c https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt -P ./models/vae/


# Loras
#!wget -c https://civitai.com/api/download/models/10350 -O ./models/loras/theovercomer8sContrastFix_sd21768.safetensors #theovercomer8sContrastFix SD2.x 768-v
#!wget -c https://civitai.com/api/download/models/10638 -O ./models/loras/theovercomer8sContrastFix_sd15.safetensors #theovercomer8sContrastFix SD1.x
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors -P ./models/loras/ #SDXL offset noise lora


# T2I-Adapter
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_depth_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_seg_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_sketch_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_keypose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_openpose_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_color_sd14v1.pth -P ./models/controlnet/
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_canny_sd14v1.pth -P ./models/controlnet/

# T2I Styles Model
#!wget -c https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_style_sd14v1.pth -P ./models/style_models/

# CLIPVision model (needed for styles model)
#!wget -c https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin -O ./models/clip_vision/clip_vit14.bin


# ControlNet
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_seg_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11u_sd15_tile_fp16.safetensors -P ./models/controlnet/

# ControlNet SDXL
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors -P ./models/controlnet/

# Controlnet Preprocessor nodes by Fannovel16
#!cd custom_nodes && git clone https://github.com/Fannovel16/comfy_controlnet_preprocessors; cd comfy_controlnet_preprocessors && python install.py


# GLIGEN
#!wget -c https://huggingface.co/comfyanonymous/GLIGEN_pruned_safetensors/resolve/main/gligen_sd14_textbox_pruned_fp16.safetensors -P ./models/gligen/


# ESRGAN upscale model
#!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth -P ./models/upscale_models/
#!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth -P ./models/upscale_models/




--2026-10-02 17:30:43--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt [following]
--2026-10-02 17:30:43--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/773d9e5ddbf6a84612066b3ea00a0bcfe4ba870a9befc88816f71b5927e76e38?X-Xet-Cas-Uid=public&user_id=public&xip=HjoOWNH85W4&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27v1-5-pruned-emaonly.ckpt%3B+filename%3D%22v1-5-pruned-em

### Run ComfyUI with cloudflared (Recommended Way)




In [4]:
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

--2026-10-02 17:30:50--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64.deb [following]
--2026-10-02 17:30:51--  https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/edb34d12-c0f8-4ce8-9d92-ccdb5e93b0eb?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-02T18%3A08%3A17Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

### Run ComfyUI with localtunnel




In [5]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 4s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸python3: can't open file '/content/drive/MyDrive/ComfyUI/main.py': [Errno 2] No such file or directory


### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [6]:
%cd /content/drive/MyDrive/ComfyUI
!pip install -q insightface onnxruntime
!npm install -g localtunnel
import subprocess
import threading
import time
import socket
import urllib.request
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

/content/drive/MyDrive/ComfyUI
⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 969ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧python3: can't open file '/content/drive/MyDrive/ComfyUI/main.py': [Errno 2] No such file or directory


In [7]:

!ls -la /content/drive/MyDrive
!ls -la /content/drive/MyDrive/ComfyUI

total 76435
drwx------ 12 root root     4096 Oct  2 15:18  .
drwx------  5 root root     4096 Oct  2 15:18  ..
drwx------  3 root root     4096 Sep 12 09:02 'Bheem Voice'
lrw-------  1 root root        0 Mar  5  2022 'bull boss1.mov' -> '/content/drive/ '
drwx------  2 root root     4096 Jul  4  2025 'Colab Notebooks'
drwx------  5 root root     4096 Oct  2 17:29  ComfyUI
drwx------  2 root root     4096 Sep  9 10:49  ComfyUI_Models
drwx------  2 root root     4096 Sep  9 12:43  ComfyUI_Output
-rw-------  1 root root  1762872 Sep  7  2025 'Document from Biswanath.pdf'
drwx------  2 root root     4096 Jun 15  2025 'Google AI Studio'
-rw-------  1 root root  3017667 Aug 24  2014 'How to get started with Drive.pdf'
drwx------  3 root root     4096 Sep 11 04:01 'local ai studio'
drwx------  2 root root     4096 Dec  9  2020  Nagu
-rw-------  1 root root      188 Jan  8  2026  SparkAppLog_7-2.gdoc
drwx------  4 root root     4096 Sep  6 15:42  storyboard_ai
drwx------  2 root root     4096 

In [8]:

!find /content/drive/MyDrive -name "sd_xl_base_1.0.safetensors" 2>/dev/null
!find /content/drive/MyDrive -name "ComfyUI" 2>/dev/null

/content/drive/MyDrive/ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors
/content/drive/MyDrive/ComfyUI


In [9]:

# REPAIR SCRIPT
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

# Link models and custom nodes from Drive to the fresh ComfyUI
!cp -rsf /content/drive/MyDrive/ComfyUI/models/* /content/ComfyUI/models/
!cp -rsf /content/drive/MyDrive/ComfyUI/custom_nodes/* /content/ComfyUI/custom_nodes/

%cd /content/ComfyUI
!pip install -r requirements.txt
!pip install -q insightface onnxruntime
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 54667, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 54667 (delta 277), reused 184 (delta 183), pack-reused 54346 (from 2)
Receiving objects: 100% (54667/54667), 98.57 MiB | 9.32 MiB/s, done.
Resolving deltas: 100% (37336/37336), done.
/content/ComfyUI
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 22 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-02 17:33:04.238
** Platf

In [10]:

!ls -la /content/drive/MyDrive/ComfyUI/models/checkpoints/

total 10940887
drwx------  2 root root       4096 Oct  2 17:30 .
drwx------ 19 root root       4096 Sep 12 14:51 ..
-rw-------  1 root root 6938078334 Oct  2 15:47 sd_xl_base_1.0.safetensors
-rw-------  1 root root 4265380512 Oct  2 17:30 v1-5-pruned-emaonly.ckpt


In [11]:

!ls -la /content/drive/MyDrive/

total 76435
drwx------ 12 root root     4096 Oct  2 15:18  .
drwx------  5 root root     4096 Oct  2 15:18  ..
drwx------  3 root root     4096 Sep 12 09:02 'Bheem Voice'
lrw-------  1 root root        0 Mar  5  2022 'bull boss1.mov' -> '/content/drive/ '
drwx------  2 root root     4096 Jul  4  2025 'Colab Notebooks'
drwx------  5 root root     4096 Oct  2 17:29  ComfyUI
drwx------  2 root root     4096 Sep  9 10:49  ComfyUI_Models
drwx------  2 root root     4096 Sep  9 12:43  ComfyUI_Output
-rw-------  1 root root  1762872 Sep  7  2025 'Document from Biswanath.pdf'
drwx------  2 root root     4096 Jun 15  2025 'Google AI Studio'
-rw-------  1 root root  3017667 Aug 24  2014 'How to get started with Drive.pdf'
drwx------  3 root root     4096 Sep 11 04:01 'local ai studio'
drwx------  2 root root     4096 Dec  9  2020  Nagu
-rw-------  1 root root      188 Jan  8  2026  SparkAppLog_7-2.gdoc
drwx------  4 root root     4096 Sep  6 15:42  storyboard_ai
drwx------  2 root root     4096 

In [12]:

%cd /content/ComfyUI
import subprocess
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
^C


In [13]:

!ls -la /content/drive/MyDrive/ComfyUI/custom_nodes/

total 12
drwx------  3 root root 4096 Sep 12 14:51 .
drwx------  5 root root 4096 Oct  2 17:29 ..
drwx------ 15 root root 4096 Oct  2 15:44 ComfyUI-Manager


In [14]:

!ls -la /content/drive/MyDrive/bull\ boss1.mov!ls -la /content/drive/MyDrive/ComfyUI/models/ipadapter/

ls: cannot access '/content/drive/MyDrive/bull boss1.mov!ls': No such file or directory
/content/drive/MyDrive/ComfyUI/models/ipadapter/:
total 1728076
drwx------  2 root root       4096 Oct  2 15:51 .
drwx------ 19 root root       4096 Sep 12 14:51 ..
-rw-------  1 root root 1071149741 Oct  2 15:51 ip-adapter-faceid_sdxl.bin
-rw-------  1 root root  698391064 Oct  2 15:47 ip-adapter_sdxl_vit-h.safetensors


In [15]:
%cd /content/ComfyUI/custom_nodes/
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git

%cd /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
!pip install -r requirements.txt

!mkdir -p /content/ComfyUI/models/ipadapter
%cd /content/ComfyUI/models/ipadapter
!wget -O ip-adapter-plus-face_sdxl_vit-h.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid_sdxl.bin

!mkdir -p /content/ComfyUI/models/clip_vision
%cd /content/ComfyUI/models/clip_vision
!wget -O clip_vision_g.safetensors https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors

%cd /content/ComfyUI
!python main.py --dont-print-serverb
%cd /content/ComfyUI
import subprocess
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI_IPAdapter_plus'...
remote: Enumerating objects: 813, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 813 (delta 404), reused 297 (delta 297), pack-reused 384 (from 2)
Receiving objects: 100% (813/813), 4.73 MiB | 10.41 MiB/s, done.
Resolving deltas: 100% (551/551), done.
Cloning into 'comfyui_controlnet_aux'...
remote: Enumerating objects: 5800, done.
remote: Counting objects: 100% (2084/2084), done.
remote: Compressing objects: 100% (487/487), done.
remote: Total 5800 (delta 1770), reused 1606 (delta 1597), pack-reused 3716 (from 4)
Receiving objects: 100% (5800/5800), 42.96 MiB | 16.20 MiB/s, done.
Resolving deltas: 100% (3157/3157), done.
/content/ComfyUI/custom_nodes/comfyui_controlnet_aux
Ignoring importlib_metadata: markers 'python_version < "3.8"' don't match your environment
/content/ComfyUI/models/ipadapter
--2026-10-02 17:36:39--  https://huggingface.co/h

In [16]:

%%writefile /content/queue_local.py
import urllib.request
import json

prompt = {
    "2": {
        "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "10": {
        "inputs": {"image": "char.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "11": {
        "inputs": {"image": "pose.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "20": {
        "inputs": {
            "model": ["2", 0],
            "preset": "PLUS FACE (portraits)"
        },
        "class_type": "IPAdapterUnifiedLoader"
    },
    "21": {
        "inputs": {
            "weight": 0.85,
            "weight_type": "linear",
            "combine_embeds": "concat",
            "start_at": 0.0,
            "end_at": 1.0,
            "embeds_scaling": "V only",
            "model": ["20", 0],
            "ipadapter": ["20", 1],
            "image": ["10", 0]
        },
        "class_type": "IPAdapterAdvanced"
    },
    "30": {
        "inputs": {
            "bbox_detector": "yolox_l.onnx",
            "pose_estimator": "dw-ll_ucoco_384.onnx",
            "image": ["11", 0],
            "resolution": 1024
        },
        "class_type": "DWPreprocessor"
    },
    "31": {
        "inputs": {"control_net_name": "OpenPoseXL2.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "32": {
        "inputs": {
            "strength": 1.0,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["80", 0],
            "negative": ["81", 0],
            "control_net": ["31", 0],
            "image": ["30", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "40": {
        "inputs": {
            "a": 6.28,
            "bg_threshold": 0.1,
            "image": ["11", 0]
        },
        "class_type": "MiDaS-DepthMapPreprocessor"
    },
    "41": {
        "inputs": {"control_net_name": "depth_sdxl.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "42": {
        "inputs": {
            "strength": 0.8,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["32", 0],
            "negative": ["32", 1],
            "control_net": ["41", 0],
            "image": ["40", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "80": {
        "inputs": {
            "text": "masterpiece, best quality, 1boy, chhota bheem character, muscular child, orange dhoti, traditional indian village background, highly detailed face, intense action pose, cinematic lighting, dramatic shadows, 4k",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "81": {
        "inputs": {
            "text": "low quality, worst quality, deformed, mutated, extra limbs, bad anatomy, text, watermark, missing fingers",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "85": {
        "inputs": {
            "width": 1920,
            "height": 1080,
            "batch_size": 1
        },
        "class_type": "EmptyLatentImage"
    },
    "86": {
        "inputs": {
            "seed": 847294,
            "steps": 30,
            "cfg": 7.0,
            "sampler_name": "dpmpp_2m",
            "scheduler": "karras",
            "denoise": 1.0,
            "model": ["21", 0],
            "positive": ["42", 0],
            "negative": ["42", 1],
            "latent_image": ["85", 0]
        },
        "class_type": "KSampler"
    },
    "95": {
        "inputs": {
            "samples": ["86", 0],
            "vae": ["2", 2]
        },
        "class_type": "VAEDecode"
    },
    "96": {
        "inputs": {
            "filename_prefix": "bheem_final",
            "images": ["95", 0]
        },
        "class_type": "SaveImage"
    }
}

p = {"prompt": prompt}
data = json.dumps(p).encode('utf-8')
req =  urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
req.add_header('Content-Type', 'application/json')
with urllib.request.urlopen(req) as response:
    print(response.read().decode('utf-8'))


Overwriting /content/queue_local.py


In [17]:

!python /content/queue_local.py

Traceback (most recent call last):
  File "/usr/lib/python3.13/urllib/request.py", line 1318, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              encode_chunked=req.has_header('Transfer-encoding'))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/http/client.py", line 1386, in request
    self._send_request(method, url, body, headers, encode_chunked)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/http/client.py", line 1432, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/http/client.py", line 1381, in endheaders
    self._send_output(message_body, encode_chunked=encode_chunked)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/http/client.p

In [18]:

import subprocess
import time
import urllib.request
import json

print("Starting ComfyUI in background...")
subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd="/content/ComfyUI")

print("Waiting for server to start (15 seconds)...")
time.sleep(15)

print("Queueing prompt...")
prompt = {
    "2": {
        "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "10": {
        "inputs": {"image": "char.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "11": {
        "inputs": {"image": "pose.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "20": {
        "inputs": {
            "model": ["2", 0],
            "preset": "PLUS FACE (portraits)"
        },
        "class_type": "IPAdapterUnifiedLoader"
    },
    "21": {
        "inputs": {
            "weight": 0.85,
            "weight_type": "linear",
            "combine_embeds": "concat",
            "start_at": 0.0,
            "end_at": 1.0,
            "embeds_scaling": "V only",
            "model": ["20", 0],
            "ipadapter": ["20", 1],
            "image": ["10", 0]
        },
        "class_type": "IPAdapterAdvanced"
    },
    "30": {
        "inputs": {
            "bbox_detector": "yolox_l.onnx",
            "pose_estimator": "dw-ll_ucoco_384.onnx",
            "image": ["11", 0],
            "resolution": 1024
        },
        "class_type": "DWPreprocessor"
    },
    "31": {
        "inputs": {"control_net_name": "OpenPoseXL2.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "32": {
        "inputs": {
            "strength": 1.0,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["80", 0],
            "negative": ["81", 0],
            "control_net": ["31", 0],
            "image": ["30", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "40": {
        "inputs": {
            "a": 6.28,
            "bg_threshold": 0.1,
            "image": ["11", 0]
        },
        "class_type": "MiDaS-DepthMapPreprocessor"
    },
    "41": {
        "inputs": {"control_net_name": "depth_sdxl.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "42": {
        "inputs": {
            "strength": 0.8,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["32", 0],
            "negative": ["32", 1],
            "control_net": ["41", 0],
            "image": ["40", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "80": {
        "inputs": {
            "text": "masterpiece, best quality, 1boy, chhota bheem character, muscular child, orange dhoti, traditional indian village background, highly detailed face, intense action pose, cinematic lighting, dramatic shadows, 4k",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "81": {
        "inputs": {
            "text": "low quality, worst quality, deformed, mutated, extra limbs, bad anatomy, text, watermark, missing fingers",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "85": {
        "inputs": {
            "width": 1920,
            "height": 1080,
            "batch_size": 1
        },
        "class_type": "EmptyLatentImage"
    },
    "86": {
        "inputs": {
            "seed": 847294,
            "steps": 30,
            "cfg": 7.0,
            "sampler_name": "dpmpp_2m",
            "scheduler": "karras",
            "denoise": 1.0,
            "model": ["21", 0],
            "positive": ["42", 0],
            "negative": ["42", 1],
            "latent_image": ["85", 0]
        },
        "class_type": "KSampler"
    },
    "95": {
        "inputs": {
            "samples": ["86", 0],
            "vae": ["2", 2]
        },
        "class_type": "VAEDecode"
    },
    "96": {
        "inputs": {
            "filename_prefix": "bheem_final",
            "images": ["95", 0]
        },
        "class_type": "SaveImage"
    }
}

p = {"prompt": prompt}
data = json.dumps(p).encode('utf-8')
try:
    req =  urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    req.add_header('Content-Type', 'application/json')
    with urllib.request.urlopen(req) as response:
        print("Queue response:", response.read().decode('utf-8'))
except Exception as e:
    print("Error:", e)


Starting ComfyUI in background...
Waiting for server to start (15 seconds)...


KeyboardInterrupt: 

In [ ]:

!ls -la /content/ComfyUI/output/

In [ ]:


import subprocess
import time
import urllib.request
import json
import os

print("Killing existing python processes...")
os.system("pkill -9 python")
time.sleep(2)

print("Starting ComfyUI in background...")
proc = subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd="/content/ComfyUI")

print("Waiting for server to start...")
for i in range(30):
    time.sleep(5)
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                print("Server is up!")
                break
    except Exception:
        print(f"Waiting... ({i}/30)")

print("Queueing prompt...")
prompt = {
    "2": {
        "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "10": {
        "inputs": {"image": "char.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "11": {
        "inputs": {"image": "pose.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "20": {
        "inputs": {
            "model": ["2", 0],
            "preset": "PLUS FACE (portraits)"
        },
        "class_type": "IPAdapterUnifiedLoader"
    },
    "21": {
        "inputs": {
            "weight": 0.85,
            "weight_type": "linear",
            "combine_embeds": "concat",
            "start_at": 0.0,
            "end_at": 1.0,
            "embeds_scaling": "V only",
            "model": ["20", 0],
            "ipadapter": ["20", 1],
            "image": ["10", 0]
        },
        "class_type": "IPAdapterAdvanced"
    },
    "30": {
        "inputs": {
            "bbox_detector": "yolox_l.onnx",
            "pose_estimator": "dw-ll_ucoco_384.onnx",
            "image": ["11", 0],
            "resolution": 1024
        },
        "class_type": "DWPreprocessor"
    },
    "31": {
        "inputs": {"control_net_name": "OpenPoseXL2.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "32": {
        "inputs": {
            "strength": 1.0,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["80", 0],
            "negative": ["81", 0],
            "control_net": ["31", 0],
            "image": ["30", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "40": {
        "inputs": {
            "a": 6.28,
            "bg_threshold": 0.1,
            "image": ["11", 0]
        },
        "class_type": "MiDaS-DepthMapPreprocessor"
    },
    "41": {
        "inputs": {"control_net_name": "depth_sdxl.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "42": {
        "inputs": {
            "strength": 0.8,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["32", 0],
            "negative": ["32", 1],
            "control_net": ["41", 0],
            "image": ["40", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "80": {
        "inputs": {
            "text": "masterpiece, best quality, 1boy, chhota bheem character, muscular child, orange dhoti, traditional indian village background, highly detailed face, intense action pose, cinematic lighting, dramatic shadows, 4k",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "81": {
        "inputs": {
            "text": "low quality, worst quality, deformed, mutated, extra limbs, bad anatomy, text, watermark, missing fingers",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "85": {
        "inputs": {
            "width": 1920,
            "height": 1080,
            "batch_size": 1
        },
        "class_type": "EmptyLatentImage"
    },
    "86": {
        "inputs": {
            "seed": 847294,
            "steps": 30,
            "cfg": 7.0,
            "sampler_name": "dpmpp_2m",
            "scheduler": "karras",
            "denoise": 1.0,
            "model": ["21", 0],
            "positive": ["42", 0],
            "negative": ["42", 1],
            "latent_image": ["85", 0]
        },
        "class_type": "KSampler"
    },
    "95": {
        "inputs": {
            "samples": ["86", 0],
            "vae": ["2", 2]
        },
        "class_type": "VAEDecode"
    },
    "96": {
        "inputs": {
            "filename_prefix": "bheem_final",
            "images": ["95", 0]
        },
        "class_type": "SaveImage"
    }
}

p = {"prompt": prompt}
data = json.dumps(p).encode('utf-8')
try:
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    req.add_header('Content-Type', 'application/json')
    with urllib.request.urlopen(req) as response:
        print("Queue response:", response.read().decode('utf-8'))

    print("Monitoring execution...")
    while True:
        try:
            req_hist = urllib.request.Request("http://127.0.0.1:8188/history")
            with urllib.request.urlopen(req_hist) as res_hist:
                hist = json.loads(res_hist.read().decode('utf-8'))
                if hist:
                    print("Generation complete!")
                    break
        except Exception:
            pass
        time.sleep(5)

except Exception as e:
    print("Error:", e)

import subprocess
import time
import urllib.request
import json
import os

print("Killing existing python processes...")
os.system("pkill -9 python")
time.sleep(2)

print("Starting ComfyUI in background...")
proc = subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd="/content/ComfyUI")

print("Waiting for server to start...")
for i in range(40):
    time.sleep(5)
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                print("Server is up!")
                break
    except Exception:
        print(f"Waiting... ({i}/40)")

print("Queueing 1024x1024 prompt...")
prompt = {
    "2": {
        "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "10": {
        "inputs": {"image": "char.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "11": {
        "inputs": {"image": "pose.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "20": {
        "inputs": {
            "model": ["2", 0],
            "preset": "PLUS FACE (portraits)"
        },
        "class_type": "IPAdapterUnifiedLoader"
    },
    "21": {
        "inputs": {
            "weight": 0.85,
            "weight_type": "linear",
            "combine_embeds": "concat",
            "start_at": 0.0,
            "end_at": 1.0,
            "embeds_scaling": "V only",
            "model": ["20", 0],
            "ipadapter": ["20", 1],
            "image": ["10", 0]
        },
        "class_type": "IPAdapterAdvanced"
    },
    "30": {
        "inputs": {
            "bbox_detector": "yolox_l.onnx",
            "pose_estimator": "dw-ll_ucoco_384.onnx",
            "image": ["11", 0],
            "resolution": 1024
        },
        "class_type": "DWPreprocessor"
    },
    "31": {
        "inputs": {"control_net_name": "OpenPoseXL2.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "32": {
        "inputs": {
            "strength": 0.8,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["80", 0],
            "negative": ["81", 0],
            "control_net": ["31", 0],
            "image": ["30", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "40": {
        "inputs": {
            "a": 6.28,
            "bg_threshold": 0.1,
            "image": ["11", 0]
        },
        "class_type": "MiDaS-DepthMapPreprocessor"
    },
    "41": {
        "inputs": {"control_net_name": "depth_sdxl.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "42": {
        "inputs": {
            "strength": 0.7,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["32", 0],
            "negative": ["32", 1],
            "control_net": ["41", 0],
            "image": ["40", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "80": {
        "inputs": {
            "text": "masterpiece, best quality, 1boy, chhota bheem character, muscular child, orange dhoti, traditional indian village background, highly detailed face, intense action pose, cinematic lighting, dramatic shadows, 4k",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "81": {
        "inputs": {
            "text": "low quality, worst quality, deformed, mutated, extra limbs, bad anatomy, text, watermark, missing fingers",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "85": {
        "inputs": {
            "width": 1024,
            "height": 1024,
            "batch_size": 1
        },
        "class_type": "EmptyLatentImage"
    },
    "86": {
        "inputs": {
            "seed": 847294,
            "steps": 25,
            "cfg": 7.0,
            "sampler_name": "dpmpp_2m",
            "scheduler": "karras",
            "denoise": 1.0,
            "model": ["21", 0],
            "positive": ["42", 0],
            "negative": ["42", 1],
            "latent_image": ["85", 0]
        },
        "class_type": "KSampler"
    },
    "95": {
        "inputs": {
            "samples": ["86", 0],
            "vae": ["2", 2]
        },
        "class_type": "VAEDecode"
    },
    "96": {
        "inputs": {
            "filename_prefix": "bheem_final1024",
            "images": ["95", 0]
        },
        "class_type": "SaveImage"
    }
}

p = {"prompt": prompt}
data = json.dumps(p).encode('utf-8')
try:
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    req.add_header('Content-Type', 'application/json')
    with urllib.request.urlopen(req) as response:
        res = json.loads(response.read().decode('utf-8'))
        prompt_id = res.get('prompt_id')
        print("Queue response prompt_id:", prompt_id)

    if prompt_id:
        print("Monitoring execution...")
        while True:
            try:
                req_hist = urllib.request.Request("http://127.0.0.1:8188/history/" + prompt_id)
                with urllib.request.urlopen(req_hist) as res_hist:
                    hist = json.loads(res_hist.read().decode('utf-8'))
                    if prompt_id in hist:
                        print("Generation complete!")

                        import shutil
                        # Copy to Drive
                        out_dir = "/content/ComfyUI/output"
                        drive_dir = "/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em"

                        for f in os.listdir(out_dir):
                            if f.startswith("bheem_final1024") and f.endswith(".png"):
                                print(f"Copying {f} to Google Drive...")
                                shutil.copy(os.path.join(out_dir, f), drive_dir)

                        break
            except Exception as e:
                pass
            time.sleep(5)

except Exception as e:
    print("Error:", e)


In [ ]:

import os
import base64

print("Checking output directory...")
out_dir = "/content/ComfyUI/output"
if os.path.exists(out_dir):
    files = os.listdir(out_dir)
    print("Files:", files)

    for f in files:
        if f.startswith("bheem_final") and f.endswith(".png"):
            print("Found image:", f)
            with open(os.path.join(out_dir, f), "rb") as img:
                b64 = base64.b64encode(img.read()).decode("utf-8")
                print("BASE64_START")
                print(b64)
                print("BASE64_END")
            break
else:
    print("No output dir")

import urllib.request
try:
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    with urllib.request.urlopen(req, timeout=2) as response:
        print("ComfyUI Server is ALIVE:", response.status)
except Exception as e:
    print("ComfyUI Server is DEAD:", e)


In [ ]:

import os
print("Checking output folder:")
if os.path.exists("/content/ComfyUI/output"):
    print(os.listdir("/content/ComfyUI/output"))
else:
    print("No output folder")

import subprocess
print("Checking ComfyUI process:")
os.system("ps aux | grep python")

import subprocess
import time
import urllib.request
import json
import os
import shutil

print("Killing existing python processes...")
os.system("pkill -9 python")
os.system("pkill -9 lt")
time.sleep(2)

print("Starting ComfyUI in background with --lowvram...")
proc = subprocess.Popen(["python", "main.py", "--dont-print-server", "--lowvram"], cwd="/content/ComfyUI")

print("Waiting for server to start...")
for i in range(40):
    time.sleep(5)
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                print("Server is up!")
                break
    except Exception:
        print(f"Waiting... ({i}/40)")

print("Queueing 768x768 prompt with BOTH ControlNets...")
prompt = {
    "2": {
        "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "10": {
        "inputs": {"image": "char.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "11": {
        "inputs": {"image": "pose.jpg", "upload": "image"},
        "class_type": "LoadImage"
    },
    "20": {
        "inputs": {
            "model": ["2", 0],
            "preset": "PLUS FACE (portraits)"
        },
        "class_type": "IPAdapterUnifiedLoader"
    },
    "21": {
        "inputs": {
            "weight": 0.85,
            "weight_type": "linear",
            "combine_embeds": "concat",
            "start_at": 0.0,
            "end_at": 1.0,
            "embeds_scaling": "V only",
            "model": ["20", 0],
            "ipadapter": ["20", 1],
            "image": ["10", 0]
        },
        "class_type": "IPAdapterAdvanced"
    },
    "30": {
        "inputs": {
            "bbox_detector": "yolox_l.onnx",
            "pose_estimator": "dw-ll_ucoco_384.onnx",
            "image": ["11", 0],
            "resolution": 768
        },
        "class_type": "DWPreprocessor"
    },
    "31": {
        "inputs": {"control_net_name": "OpenPoseXL2.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "32": {
        "inputs": {
            "strength": 0.8,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["80", 0],
            "negative": ["81", 0],
            "control_net": ["31", 0],
            "image": ["30", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "40": {
        "inputs": {
            "a": 6.28,
            "bg_threshold": 0.1,
            "image": ["11", 0]
        },
        "class_type": "MiDaS-DepthMapPreprocessor"
    },
    "41": {
        "inputs": {"control_net_name": "depth_sdxl.safetensors"},
        "class_type": "ControlNetLoader"
    },
    "42": {
        "inputs": {
            "strength": 0.7,
            "start_percent": 0.0,
            "end_percent": 1.0,
            "positive": ["32", 0],
            "negative": ["32", 1],
            "control_net": ["41", 0],
            "image": ["40", 0]
        },
        "class_type": "ControlNetApplyAdvanced"
    },
    "80": {
        "inputs": {
            "text": "masterpiece, best quality, 1boy, chhota bheem character, muscular child, orange dhoti, traditional indian village background, highly detailed face, intense action pose, cinematic lighting, dramatic shadows, 4k",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "81": {
        "inputs": {
            "text": "low quality, worst quality, deformed, mutated, extra limbs, bad anatomy, text, watermark, missing fingers",
            "clip": ["2", 1]
        },
        "class_type": "CLIPTextEncode"
    },
    "85": {
        "inputs": {
            "width": 768,
            "height": 768,
            "batch_size": 1
        },
        "class_type": "EmptyLatentImage"
    },
    "86": {
        "inputs": {
            "seed": 29384,
            "steps": 25,
            "cfg": 7.0,
            "sampler_name": "dpmpp_2m",
            "scheduler": "karras",
            "denoise": 1.0,
            "model": ["21", 0],
            "positive": ["42", 0],
            "negative": ["42", 1],
            "latent_image": ["85", 0]
        },
        "class_type": "KSampler"
    },
    "95": {
        "inputs": {
            "samples": ["86", 0],
            "vae": ["2", 2]
        },
        "class_type": "VAEDecode"
    },
    "96": {
        "inputs": {
            "filename_prefix": "bheem_final768",
            "images": ["95", 0]
        },
        "class_type": "SaveImage"
    }
}

p = {"prompt": prompt}
data = json.dumps(p).encode('utf-8')
try:
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    req.add_header('Content-Type', 'application/json')
    with urllib.request.urlopen(req) as response:
        res = json.loads(response.read().decode('utf-8'))
        prompt_id = res.get('prompt_id')
        print("Queue response prompt_id:", prompt_id)

    if prompt_id:
        print("Monitoring execution...")
        while True:
            try:
                req_hist = urllib.request.Request("http://127.0.0.1:8188/history/" + prompt_id)
                with urllib.request.urlopen(req_hist) as res_hist:
                    hist = json.loads(res_hist.read().decode('utf-8'))
                    if prompt_id in hist:
                        print("Generation complete!")

                        # Copy to Drive
                        out_dir = "/content/ComfyUI/output"
                        drive_dir = "/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em"

                        for f in os.listdir(out_dir):
                            if f.startswith("bheem_final768") and f.endswith(".png"):
                                print(f"Copying {f} to Google Drive...")
                                shutil.copy(os.path.join(out_dir, f), drive_dir)

                                # Export base64 to a text file in drive so I can curl it easily later!
                                import base64
                                b64_path = os.path.join(drive_dir, "bheem_final768_b64.txt")
                                with open(os.path.join(out_dir, f), "rb") as img:
                                    b64 = base64.b64encode(img.read()).decode("utf-8")
                                    with open(b64_path, "w") as b64f:
                                        b64f.write(b64)
                                print("Wrote base64 to", b64_path)

                        break
            except Exception as e:
                pass
            time.sleep(5)

except Exception as e:
    print("Error:", e)

import subprocess
import time
import urllib.request
import os

print("Killing existing processes...")
os.system("pkill -9 python")
os.system("pkill -9 lt")
time.sleep(2)

print("Starting ComfyUI...")
proc = subprocess.Popen(["python", "main.py", "--lowvram"], cwd="/content/ComfyUI")

print("Waiting for server to start...")
for i in range(20):
    time.sleep(5)
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                print("Server is up!")
                break
    except Exception:
        pass

print("Starting Localtunnel...")
lt_proc = subprocess.Popen(["lt", "--port", "8188"], stdout=subprocess.PIPE, text=True)

url = ""
for line in lt_proc.stdout:
    if "your url is:" in line:
        url = line.split("your url is:")[1].strip()
        break

print("=======================================")
print(f"COMFYUI URL: {url}")
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"LOCALTUNNEL PASSWORD: {ip}")
print("=======================================")

# Keep the cell running
lt_proc.wait()

# RAM MANAGEMENT SYSTEM (Virtual RAM / Swap)
import os
print("Creating 12GB Swap File for Extra RAM...")
os.system("fallocate -l 12G /swapfile")
os.system("chmod 600 /swapfile")
os.system("mkswap /swapfile")
os.system("swapon /swapfile")
print("Swap RAM created successfully! No more OOM crashes!")
!free -h

import os
os.system("fallocate -l 12G /swapfile")
os.system("chmod 600 /swapfile")
os.system("mkswap /swapfile")
os.system("swapon /swapfile")
print("Swap RAM created successfully! No more OOM crashes!")


In [ ]:


import subprocess
import time
import urllib.request
import os

print("Starting ComfyUI...")
proc = subprocess.Popen(["python", "main.py", "--lowvram"], cwd="/content/ComfyUI")

print("Waiting for server to start...")
for i in range(20):
    time.sleep(5)
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                print("Server is up!")
                break
    except Exception:
        pass

print("Starting Localtunnel...")
lt_proc = subprocess.Popen(["lt", "--port", "8188"], stdout=subprocess.PIPE, text=True)

url = ""
for line in lt_proc.stdout:
    if "your url is:" in line:
        url = line.split("your url is:")[1].strip()
        break

print("=======================================")
print(f"COMFYUI URL: {url}")
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"LOCALTUNNEL PASSWORD: {ip}")
print("=======================================")
lt_proc.wait()

import subprocess
import urllib.request
import os
import time

os.system("pkill -9 lt")
time.sleep(1)

lt_proc = subprocess.Popen(["lt", "--port", "8188"], stdout=subprocess.PIPE, text=True)

url = ""
for line in lt_proc.stdout:
    if "your url is:" in line:
        url = line.split("your url is:")[1].strip()
        break

ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

with open("/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em/url.txt", "w") as f:
    f.write(url + "\n" + ip)

print("Saved to url.txt!")

import subprocess
import urllib.request
import os
import time

os.system("pkill -9 lt")
time.sleep(1)

lt_proc = subprocess.Popen(["lt", "--port", "8188"], stdout=subprocess.PIPE, text=True)

url = ""
for line in lt_proc.stdout:
    if "your url is:" in line:
        url = line.split("your url is:")[1].strip()
        break

ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

with open("/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em/url.txt", "w") as f:
    f.write(url + "\n" + ip)

print("Saved to url.txt!")

import os
import time
import urllib.request
import re

print("Killing old lt...")
os.system("pkill -9 lt")
time.sleep(2)

print("Starting new lt...")
os.system("lt --port 8188 > /content/lt.log 2>&1 &")
time.sleep(5)

with open("/content/lt.log", "r") as f:
    log = f.read()

url = ""
match = re.search(r"your url is: (https://.*)", log)
if match:
    url = match.group(1).strip()

print("Getting IP...")
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print("Writing to drive...")
with open("/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em/url.txt", "w") as f:
    f.write("URL: " + url + "\nIP: " + ip)

print("Saved to url.txt!")
print("URL:", url)
print("IP:", ip)

import os
print("IP:")
os.system("curl -s ifconfig.me")
print("\nURL:")
os.system("cat /content/lt.log")

with open("/content/lt.log", "r") as f:
    log = f.read()
import re
url = ""
match = re.search(r"your url is: (https://.*)", log)
if match:
    url = match.group(1).strip()

ip = os.popen("curl -s ifconfig.me").read().strip()
with open("/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em/url.txt", "w") as f:
    f.write("URL: " + url + "\nIP: " + ip)
print("\nSaved!")

import os
import time
import urllib.request
import re

print("Killing old lt...")
os.system("pkill -9 lt")
time.sleep(2)

print("Starting new lt...")
os.system("lt --port 8188 > /content/lt.log 2>&1 &")
time.sleep(5)

with open("/content/lt.log", "r") as f:
    log = f.read()

url = ""
match = re.search(r"your url is: (https://.*)", log)
if match:
    url = match.group(1).strip()

print("Getting IP...")
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print("Writing to drive...")
with open("/content/drive/MyDrive/1FxVggsVFPIONpVHbrkQzvsaBw2HMy9Em/url.txt", "w") as f:
    f.write("URL: " + url + "\nIP: " + ip)

print("Saved to url.txt!")
print("URL:", url)
print("IP:", ip)


Starting ComfyUI...
Waiting for server to start...
Server is up!
Starting Localtunnel...
COMFYUI URL: https://dull-comics-tickle.loca.lt
LOCALTUNNEL PASSWORD: 34.125.4.247
